# 🇮🇳🇺🇸 US–India Trade Deal: What Is the Community Saying?
### Sentiment & topic analysis of real X (Twitter) posts — classic NLP (TF-IDF) vs. a Large Language Model

---

**Goal.** Collect real social-media posts about the US–India trade deal, then answer three questions:

1. **What is the overall sentiment?** Are people positive, negative or neutral about the deal?
2. **What are people talking about?** Which themes dominate the conversation?
3. **Does an LLM read the conversation differently from a classic TF-IDF model?**

**The pipeline**

```
 Raw tweets ─► Cleaning ─► Part A: Classic NLP ───────────────┐
 (X, 2,804)                • TF-IDF + hand-crafted features     │
                           • VADER sentiment (lexicon)          ├─► Part C: Compare ─► Insights
                           • Logistic Regression model          │
                           • NMF topics                         │
                       └─► Part B: LLM (Groq) ─────────────────┘
                           • sentiment + topic + sarcasm per tweet
                           • written summary of community themes
```

**Contents**

| # | Section | What happens |
|---|---|---|
| – | **Architecture** | Diagram of the whole pipeline and what each part does |
| 0 | Setup | Install and import libraries, set chart style |
| 1 | Load the data | Read the scraped tweets and inspect them |
| 2 | Clean the data | Remove duplicates, spam and off-topic long posts |
| 3 | Explore | Volume over time, most-liked posts |
| 4 | **Part A – TF-IDF** | Keywords, **feature engineering**, sentiment model, topics |
| 5 | **Part B – LLM** | An LLM (via Groq) labels every tweet and writes a summary |
| 6 | **Part C – Comparison** | Where the two approaches agree and disagree |
| 7 | Insights | What the community is talking about |
| 8 | Conclusion | Findings, limitations, next steps |

**How to run.** `Runtime ▸ Run all`. You will be asked to upload `x_all.csv`, and Part B needs a free Groq API key (see 5.1).
Parts A and C need no key and run in under a minute.

> **About the data.** Posts were collected from X's search ("Latest" tab) for tweets that mention India, the US
> and trade/tariffs, between **20 Sep and 8 Oct 2026**. Reddit was planned as a second source, but its API access
> was not available, so this analysis uses X only. The notebook works unchanged if Reddit rows are added later.

---
## 🏗️ Project architecture
The project is a **5-stage pipeline**. Data flows left to right, and after cleaning it splits into two
independent analyses (classic NLP and LLM) that meet again in the comparison.

In [ ]:
# Draw the architecture diagram (also saved as architecture.png for slides)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 6.2))
fig.patch.set_facecolor("#fcfcfb")
ax.set_xlim(0, 16)
ax.set_ylim(0, 7.2)
ax.axis("off")

STYLE = {  # fill, border
    "main": ("#ffffff", "#898781"),
    "A":    ("#fdeee6", "#eb6834"),   # Part A: classic NLP
    "B":    ("#e6f5ef", "#1baf7a"),   # Part B: LLM
}

def box(x, y, title, subtitle, kind="main", w=2.0, h=1.25):
    fill, edge = STYLE[kind]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor=fill, edgecolor=edge, linewidth=1.6))
    ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", fontsize=11, fontweight="bold", color="#0b0b0b")
    ax.text(x + w / 2, y + h * 0.28, subtitle, ha="center", va="center", fontsize=8.5, color="#52514e")
    return (x, y, w, h)

def arrow(a, b, start="right", end="left"):
    def point(bx, side):
        x, y, w, h = bx
        return {"right": (x + w, y + h / 2), "left": (x, y + h / 2),
                "top": (x + w / 2, y + h), "bottom": (x + w / 2, y)}[side]
    ax.annotate("", xy=point(b, end), xytext=point(a, start),
                arrowprops=dict(arrowstyle="-|>", color="#52514e", lw=1.4, shrinkA=2, shrinkB=2,
                                connectionstyle="angle,angleA=0,angleB=90" if start != end and end in ("top", "bottom") else "arc3"))

# Stage labels along the top
for x, label in [(1.15, "1 · COLLECT"), (3.55, "2 · STORE"), (5.95, "3 · PREPARE"),
                 (9.45, "4 · ANALYSE (two ways)"), (12.85, "5 · COMPARE"), (15.0, "6 · REPORT")]:
    ax.text(x, 6.95, label, ha="center", va="center", fontsize=9, fontweight="bold", color="#898781")

source   = box(0.15, 2.95, "X (Twitter)", "India + US + trade")
scraper  = box(2.55, 2.95, "Scraper", "twscrape · 2,804 posts")
cleaning = box(4.95, 2.95, "Cleaning", "5 rules → 1,666 posts")

fe       = box(7.35, 4.95, "Features", "TF-IDF + 12 hand-crafted", "A")
classic  = box(9.75, 4.95, "Classic models", "VADER · LogReg · NMF", "A")
prompt   = box(7.35, 0.95, "Prompt + batching", "30 posts · JSON format", "B")
llm      = box(9.75, 0.95, "Groq API", "Llama 3.1 8B → JSON", "B")

compare  = box(12.15, 2.95, "Comparison", "agreement · kappa")
report   = box(14.4, 2.95, "Insights", "charts · LLM report", w=1.5)

arrow(source, scraper); arrow(scraper, cleaning)
arrow(cleaning, fe, "top", "left"); arrow(cleaning, prompt, "bottom", "left")
arrow(fe, classic); arrow(prompt, llm)
arrow(classic, compare, "right", "top"); arrow(llm, compare, "right", "bottom")
arrow(compare, report)

ax.text(8.55, 6.4, "Part A · Classic NLP (word counting)", ha="center", fontsize=10, fontweight="bold", color="#eb6834")
ax.text(8.55, 0.45, "Part B · LLM (reading meaning)", ha="center", fontsize=10, fontweight="bold", color="#1baf7a")
ax.text(3.55, 2.55, "x_all.csv", ha="center", fontsize=8.5, color="#898781", style="italic")

fig.savefig("architecture.png", dpi=200, bbox_inches="tight")
plt.show()

### What each component does

| Stage | Component | Tool | Input → Output | Key design decision |
|---|---|---|---|---|
| 1 Collect | **Scraper** | Python, `twscrape` | X search → raw posts | Search needs India **and** US **and** trade words; usernames are hashed for privacy |
| 2 Store | **Raw dataset** | CSV | 2,804 posts, 13 columns | One flat file = easy to share and re-run |
| 3 Prepare | **Cleaning** | pandas | 2,804 → 1,666 posts | Remove duplicates, spam accounts and long newsletters, so one row = one real voice |
| 4A Analyse | **Feature engineering** | scikit-learn | text → numbers | TF-IDF (what words) + 12 hand-crafted features (how it is written) |
| 4A Analyse | **Classic models** | VADER, Logistic Regression, NMF | features → sentiment + topics | Fast, free, fully explainable |
| 4B Analyse | **Prompt + batching** | Python | posts → batches of 30 | Fixed labels and JSON format; batching keeps us inside the free limits |
| 4B Analyse | **LLM** | Groq API, Llama 3.1 8B | batch → JSON labels | Reads meaning (sarcasm, news vs. opinion); results cached so re-runs are free |
| 5 Compare | **Comparison** | scikit-learn metrics | both label sets → agreement | Accuracy, Cohen's kappa, confusion matrix, reading disagreements |
| 6 Report | **Insights** | matplotlib, LLM summary | numbers → story | Charts + a written summary of what the community is saying |

### Explaining it in 30 seconds
> "We **collect** real posts from X about the US–India trade deal with a Python scraper, **clean** them down to
> 1,666 distinct voices, and then analyse them **two ways**. Path A is classic NLP: we **engineer features**
> (TF-IDF word scores plus style features like length and exclamation marks), label sentiment with VADER, train a
> Logistic Regression model and find topics with NMF. Path B sends the same posts to an **LLM through the Groq API**,
> which returns sentiment, topic and sarcasm as JSON. Finally we **compare** the two paths and turn the results into insights."

---
## 0. Setup
Install the two libraries we need (`vaderSentiment`, and `groq` for the LLM), import everything, and define one
chart style so every figure looks consistent.

In [ ]:
!pip install -q vaderSentiment groq

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown

# Machine learning (scikit-learn)
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.decomposition import NMF
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score, confusion_matrix

# Rule-based sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42   # fixed seed → same results every run

In [ ]:
# ---- One colour scheme for the whole notebook --------------------------------
# Sentiment is a "polarity" scale, so it gets two opposite colours with grey in the middle.
SENTIMENTS  = ["Negative", "Neutral", "Positive"]
SENT_COLORS = {"Negative": "#e34948", "Neutral": "#a3a19a", "Positive": "#2a78d6"}

# Comparing methods = different "identities", so each method gets its own colour.
METHOD_COLORS = {"VADER": "#2a78d6", "TF-IDF model": "#eb6834", "LLM": "#1baf7a"}

MAIN, INK, INK2, MUTED, GRID, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
BLUES = LinearSegmentedColormap.from_list("blues", ["#f0efec", "#86b6ef", "#2a78d6", "#104281"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 13, "axes.titlepad": 12,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2, "text.color": INK,
    "font.size": 11, "figure.dpi": 110, "legend.frameon": False,
})


def share_table(data, group_col, label_col, min_posts=10):
    # % of Negative / Neutral / Positive inside each group (each row adds up to 100).
    # Groups with fewer than `min_posts` posts are left out; biggest groups come first.
    sizes = data[group_col].value_counts()
    keep = sizes[sizes >= min_posts].index
    table = pd.crosstab(data[group_col], data[label_col], normalize="index") * 100
    return table.reindex(index=keep, columns=SENTIMENTS, fill_value=0)


def plot_sentiment_shares(table, title, ax=None):
    # 100% stacked horizontal bars: one bar per group, split by sentiment.
    ax = ax or plt.subplots(figsize=(10, 0.55 * len(table) + 1.6))[1]
    left = np.zeros(len(table))
    for s in SENTIMENTS:
        values = table[s].to_numpy()
        ax.barh(table.index.astype(str), values, left=left, color=SENT_COLORS[s],
                edgecolor=SURFACE, linewidth=2, height=0.62, label=s)
        for y, (start, v) in enumerate(zip(left, values)):
            if v >= 6:   # only label segments wide enough to read
                ax.text(start + v / 2, y, f"{v:.0f}%", ha="center", va="center", fontsize=9,
                        color=INK if s == "Neutral" else "white")
        left += values
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_xlabel("% of posts")
    ax.set_title(title, pad=30)
    ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0.3)
    return ax


def plot_barh(series, title, xlabel, color=MAIN, fmt="{:.0f}"):
    # Simple ranked horizontal bar chart with the value printed at the end of each bar.
    fig, ax = plt.subplots(figsize=(10, 0.36 * len(series) + 1.2))
    ax.barh(series.index.astype(str), series.values, color=color, height=0.65)
    for y, v in enumerate(series.values):
        ax.text(v, y, " " + fmt.format(v), va="center", fontsize=9, color=INK2)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.margins(x=0.12)
    plt.show()

---
## 1. Load the data
The CSV comes from our scraper. Each row is one post:

| column | meaning |
|---|---|
| `id` | unique post id |
| `created_utc` | when it was posted (UTC) |
| `author_hash` | the username, scrambled (hashed) for privacy |
| `text` | what the post says |
| `score` / `num_replies` | likes and replies (engagement) |
| `url` | link to the original post |

In [ ]:
DATA_PATH = "x_all.csv"

# In Colab: if the file is not there yet, open an upload dialog.
if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Please upload x_all.csv")
    DATA_PATH = next(iter(files.upload()))

raw = pd.read_csv(DATA_PATH, dtype={"id": str})
print(f"{len(raw):,} posts × {raw.shape[1]} columns")
raw[["created_utc", "text", "score", "num_replies"]].head()

### 1.1 First look at data quality
Before cleaning, we check for missing values, duplicates, very active accounts and very long posts.

In [ ]:
print("Missing text            :", raw["text"].isna().sum())
print("Duplicate ids           :", raw["id"].duplicated().sum())
print("Duplicate texts         :", raw["text"].duplicated().sum())
print("Posts longer than 1,000 :", (raw["text"].str.len() > 1000).sum())
print("Median length (chars)   :", int(raw["text"].str.len().median()))

print("\nMost active accounts (posts per author):")
print(raw["author_hash"].value_counts().head(5).to_string())

**What we see**
- **Exact duplicates.** The same text is posted many times (copy-paste news alerts and bots).
- **One account dominates.** It posted the same long "daily update" hundreds of times. Left in, a single voice would distort "the community".
- **Very long posts.** Newsletters and earnings reports that mention India–US trade in one line out of fifty.

---
## 2. Clean the data
Each step has one clear reason:

| Step | What we do | Why |
|---|---|---|
| 1 | Keep posts from 20 Sep 2026 onwards | The search covers 20 Sep – 8 Oct; a few older stray posts are removed |
| 2 | Drop exact duplicate texts | Copy-pasted posts would be counted many times |
| 3 | Drop posts longer than 1,000 characters | Newsletters/reports where the topic is only a passing mention |
| 4 | Keep at most 3 posts per author (their most-liked) | Stops a few very active accounts from dominating |
| 5 | Drop near-duplicates | Same text with different @mentions or links |

In [ ]:
MAX_PER_AUTHOR = 3
MAX_LENGTH = 1000
cleaning_log = []

def log_step(step, data):
    cleaning_log.append({"step": step, "posts_left": len(data)})

df = raw[["id", "created_utc", "author_hash", "text", "score", "num_replies", "url"]].copy()
df["created_utc"] = pd.to_datetime(df["created_utc"], utc=True, format="ISO8601")
log_step("Raw data", df)

# 1. Collection window
df = df[df["created_utc"] >= pd.Timestamp("2026-09-20", tz="UTC")]
log_step("1. In collection window", df)

# 2. Exact duplicates
df = df.drop_duplicates(subset="text")
log_step("2. No exact duplicates", df)

# 3. Very long posts
df = df[df["text"].str.len() <= MAX_LENGTH]
log_step(f"3. ≤ {MAX_LENGTH:,} characters", df)

# 4. Cap posts per author (keep each author's most-liked posts)
df = df.sort_values("score", ascending=False).groupby("author_hash").head(MAX_PER_AUTHOR)
log_step(f"4. ≤ {MAX_PER_AUTHOR} posts per author", df)

# 5. Near-duplicates: compare texts after removing @mentions, links, numbers and punctuation
fingerprint = (df["text"].str.lower()
               .str.replace(r"@\w+|https?://\S+|[^a-z ]", " ", regex=True)
               .str.split().str.join(" "))
df = df[~fingerprint.duplicated()]
log_step("5. No near-duplicates", df)

df = df.sort_values("created_utc").reset_index(drop=True)
pd.DataFrame(cleaning_log)

In [ ]:
log = pd.DataFrame(cleaning_log).set_index("step")["posts_left"]
plot_barh(log, "Cleaning keeps the posts that represent real, distinct voices",
          "posts remaining after each step", fmt="{:,.0f}")

### 2.1 Two versions of the text
Different methods need different text:

- **Original text** → used by **VADER** and the **LLM**. They understand capital letters, "!!!", emojis and negation, so we leave them in.
- **Clean text** → used by **TF-IDF**. It only counts words, so we lowercase and remove links, @mentions, numbers, emojis and punctuation.

In [ ]:
def clean_for_tfidf(text):
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)   # links
    text = re.sub(r"@\w+", " ", text)           # @mentions
    text = text.replace("#", " ")               # keep hashtag words, drop the # symbol
    text = re.sub(r"[^a-z\s]", " ", text)       # numbers, emojis, punctuation
    return re.sub(r"\s+", " ", text).strip()    # extra spaces

df["clean_text"] = df["text"].apply(clean_for_tfidf)

# Before / after for three examples
df[["text", "clean_text"]].sample(3, random_state=1)

---
## 3. Explore the data
How much people posted each day, and which posts got the most attention.

In [ ]:
daily = df.set_index("created_utc").resample("D").size()
daily = daily[daily.index >= daily[daily >= 10].index.min()]   # start at the first busy day

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(daily.index, daily.values, color=MAIN, width=0.75)
ax.set_title("Posts per day")
ax.set_ylabel("posts")
ax.grid(axis="x", visible=False)
fig.autofmt_xdate()
plt.show()

print(f"Period: {df['created_utc'].min():%d %b %Y} → {df['created_utc'].max():%d %b %Y} | "
      f"{len(df):,} posts | busiest day: {daily.idxmax():%d %b} ({daily.max()} posts)")

In [ ]:
# The 8 most-liked posts: what reached the most people?
df.nlargest(8, "score")[["created_utc", "score", "text"]]

---
# Part A — Classic NLP with TF-IDF
## 4.1 What is TF-IDF? (a tiny example)
A computer cannot read words, so we turn each post into numbers. **TF-IDF** gives every word a score:

- **TF (term frequency):** how often the word appears *in this post*.
- **IDF (inverse document frequency):** how *rare* the word is across *all* posts.

**TF-IDF = TF × IDF.** A word scores high when it is frequent in one post but rare overall, which makes it a
good keyword. Words that appear everywhere ("india", "trade") score low because they do not tell posts apart.

> 🧠 **The logic in one line:** `TF-IDF(word, post) = (times the word appears in the post) × log(total posts ÷ posts containing the word)`.
> The log shrinks the score of common words. scikit-learn also adds smoothing (+1) and scales every post to the same
> length, so long and short posts are comparable.

In [ ]:
toy_posts = [
    "india us trade deal talks stall",
    "india us trade deal near says minister",
    "tariffs hit india pharma exports",
]
toy = TfidfVectorizer()
toy_matrix = toy.fit_transform(toy_posts)
pd.DataFrame(toy_matrix.toarray().round(2), columns=toy.get_feature_names_out(),
             index=["post 1", "post 2", "post 3"])

**Read the table:** `india` appears in all three posts, so it gets the lowest score in each row. `pharma` and
`stall` appear once, so they get the highest. Those are the words that describe each post.

## 4.2 🛠️ Feature engineering: turning posts into numbers
A machine-learning model cannot read text. **Feature engineering** is the step where we decide *which numbers describe a
post*. Good features carry the signal (here, sentiment) and drop the noise. We build two feature sets:

| Feature set | What it captures | Example features | Built from |
|---|---|---|---|
| **1. TF-IDF text features** (4.3) | *What words* are used | `russian oil`, `plateau`, `relief`, `not` | clean text |
| **2. Hand-crafted features** (4.5) | *How* it is written and how people reacted | length, `!` count, CAPS ratio, negations, likes | original text |

Feature-engineering decisions in this notebook, and why:

| Decision | Why |
|---|---|
| Lowercase, remove links, @mentions, numbers, emojis (for TF-IDF) | "Tariff" and "tariff" should be one feature; links and handles are not words |
| Remove stop words **and** our search words | Words in every post ("the", "india", "trade") cannot tell posts apart |
| Keep negations for the sentiment features | "not good" ≠ "good" |
| Bigrams (two-word phrases) | "russian oil" means more than "russian" + "oil" separately |
| `min_df=5`, `max_df=0.6` | Drop typos/rare names and near-universal words → fewer, more reliable features |
| Count punctuation, CAPS, emojis on the **original** text | Cleaning removes them, but they carry emotion |
| `log(likes + 1)` instead of raw likes | Likes range from 0 to 4,000+; the log stops a few viral posts from dominating |
| Scale hand-crafted features (`StandardScaler`) | Puts "length in characters" and "number of !" on the same scale |
| **Do not** use the VADER score as a feature | Our labels come from VADER, so that would be *data leakage* (see 4.5) |

## 4.3 Feature set 1: TF-IDF text features
Three choices, each with a reason:

- **Stop words:** remove common English words ("the", "is") **and** our search words ("india", "us", "trade",
  "tariff"…). Every post contains the search words by design, so they carry no information.
- **Bigrams** (`ngram_range=(1, 2)`) catch two-word phrases such as *"russian oil"* or *"piyush goyal"*.
- **Two vocabularies.** Negations ("not", "no", "never") flip sentiment, so the **sentiment model keeps them**.
  For **keywords and topics** they are just noise, so that version drops them.

In [ ]:
SEARCH_WORDS = {"india", "indian", "indians", "us", "usa", "america", "american", "americans",
                "trump", "modi", "trade", "deal", "tariff", "tariffs"}
NOISE_WORDS = {"amp", "rt", "s", "t", "u", "via", "just", "like", "get", "got", "will", "says", "said", "new", "now"}
NEGATIONS = {"no", "not", "nor", "never", "cannot"}

STOP_WORDS = sorted((set(ENGLISH_STOP_WORDS) | SEARCH_WORDS | NOISE_WORDS) - NEGATIONS)

def make_tfidf(stop_words):
    return TfidfVectorizer(
        stop_words=stop_words,
        ngram_range=(1, 2),   # single words and two-word phrases
        min_df=5,             # ignore words used in fewer than 5 posts (typos, rare names)
        max_df=0.6,           # ignore words used in more than 60% of posts
    )

# For keywords and topics: negations removed
topic_vectorizer = make_tfidf(sorted(set(STOP_WORDS) | NEGATIONS))
X_topics = topic_vectorizer.fit_transform(df["clean_text"])
topic_terms = topic_vectorizer.get_feature_names_out()

# For the sentiment model: negations kept
vectorizer = make_tfidf(STOP_WORDS)
X = vectorizer.fit_transform(df["clean_text"])
terms = vectorizer.get_feature_names_out()

print(f"Keyword/topic matrix : {X_topics.shape[0]:,} posts × {X_topics.shape[1]:,} words/phrases")
print(f"Sentiment matrix     : {X.shape[0]:,} posts × {X.shape[1]:,} words/phrases")

In [ ]:
# Average TF-IDF score of each term across all posts = how central it is to the conversation
top_terms = pd.Series(np.asarray(X_topics.mean(axis=0)).ravel(), index=topic_terms).nlargest(20)
plot_barh(top_terms, "Top 20 keywords in the conversation (TF-IDF)", "average TF-IDF score", fmt="{:.3f}")

## 4.4 Sentiment labels with VADER (rule-based)
**VADER** is a dictionary of ~7,500 words with sentiment scores ("good" +1.9, "crisis" −2.4…), plus rules for
"not", "very", CAPS and "!!!". It gives every post a **compound score from −1 to +1**. We use the standard cut-offs:

| compound score | label |
|---|---|
| ≥ +0.05 | Positive |
| between | Neutral |
| ≤ −0.05 | Negative |

In [ ]:
vader = SentimentIntensityAnalyzer()
df["vader_score"] = df["text"].apply(lambda t: vader.polarity_scores(t)["compound"])
df["vader_label"] = pd.cut(df["vader_score"], bins=[-1.01, -0.05, 0.05, 1.01], labels=SENTIMENTS).astype(str)

counts = df["vader_label"].value_counts().reindex(SENTIMENTS)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(counts.index, counts.values, color=[SENT_COLORS[s] for s in counts.index], width=0.6)
for x, v in enumerate(counts.values):
    ax.text(x, v, f"{v:,}\n({v / len(df):.0%})", ha="center", va="bottom", fontsize=10, color=INK2)
ax.set_title("VADER sentiment of all posts")
ax.set_ylabel("posts")
ax.grid(axis="x", visible=False)
ax.margins(y=0.2)
plt.show()

> 🧠 **The logic.** VADER looks up each word in its dictionary, then applies simple rules: a negation ("not") flips
> the score, boosters ("very"), CAPS and "!" strengthen it, and after "but" the second half counts more.
> It adds everything up and squashes the total into −1…+1. Words it does not know (like "tariffs") count as 0, which is
> why it often misreads political posts. The demo below shows the rules in action.

In [ ]:
# How VADER thinks: one dictionary score per word, adjusted by simple rules
demo = ["good", "not good", "very good", "VERY GOOD!!!", "good, but the deal is terrible", "tariffs"]
pd.DataFrame({"phrase": demo, "compound score": [vader.polarity_scores(p)["compound"] for p in demo]})

In [ ]:
# One clear example of each label
for label in SENTIMENTS:
    row = df[df["vader_label"] == label].nlargest(1, "score").iloc[0]
    print(f"{label:8s} ({row.vader_score:+.2f}): {row.text[:220]}\n")

## 4.5 Feature set 2: hand-crafted features
TF-IDF only knows *which words* appear. These features describe *how* a post is written. Every feature has a
reason to be here:

| Feature | How it is computed | Why it might signal sentiment |
|---|---|---|
| `length_chars`, `word_count` | size of the post | Short posts are often plain headlines (Neutral); opinions run longer |
| `exclamations`, `questions` | count of `!` and `?` | Emotion, outrage, rhetorical questions |
| `caps_ratio` | share of letters in CAPITALS | "SHOUTING" = strong emotion (or a breaking-news style) |
| `emojis` | number of emojis | 🔥😡🙏 carry feeling |
| `negations` | not / no / never / don't … | Flip the meaning of nearby words |
| `hashtags`, `mentions` | `#tags`, `@users` | News accounts use hashtags; arguments happen in @replies |
| `is_reply` | post starts with `@` | Replies are conversations, often more opinionated |
| `has_percent` | contains a number with `%` | Factual posts about "100% tariffs" |
| `log_likes` | log(likes + 1) | Do emotional posts get more attention? |

**Data leakage warning.** We do **not** add the VADER score as a feature. Our labels *are* VADER labels, so the model
would just copy the score and look perfect while learning nothing. Always check that a feature is not secretly the answer.

In [ ]:
EMOJI_PATTERN = re.compile("[\U0001F300-\U0001FAFF☀-➿]")
NEGATION_PATTERN = r"\b(?:not|no|never|nor|cannot|can't|won't|don't|isn't|doesn't|didn't)\b"

text = df["text"]   # original text: punctuation, CAPS and emojis are still there
features = pd.DataFrame({
    "length_chars": text.str.len(),
    "word_count":   text.str.split().str.len(),
    "exclamations": text.str.count("!"),
    "questions":    text.str.count(r"\?"),
    "caps_ratio":   text.apply(lambda t: sum(c.isupper() for c in t) / max(1, sum(c.isalpha() for c in t))),
    "emojis":       text.str.count(EMOJI_PATTERN),
    "negations":    text.str.lower().str.count(NEGATION_PATTERN),
    "hashtags":     text.str.count(r"#\w+"),
    "mentions":     text.str.count(r"@\w+"),
    "is_reply":     text.str.startswith("@").astype(int),
    "has_percent":  text.str.contains(r"\d\s?%").astype(int),
    "log_likes":    np.log1p(df["score"]),
})
features.describe().T[["mean", "min", "max"]].round(2)

In [ ]:
# Do the features differ between Negative, Neutral and Positive posts? (average per class)
by_class = features.groupby(df["vader_label"]).mean().T[SENTIMENTS].round(2)
by_class

In [ ]:
# Compare each feature across classes as a % of the overall average (100 = average)
relative = by_class.div(features.mean(), axis=0) * 100
fig, ax = plt.subplots(figsize=(11, 5.2))
y_pos = np.arange(len(relative))
for k, s in enumerate(SENTIMENTS):
    ax.barh(y_pos + (k - 1) * 0.27, relative[s], height=0.25, color=SENT_COLORS[s], label=s)
ax.axvline(100, color=MUTED, linewidth=1)
ax.set_yticks(y_pos, relative.index)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("% of the average post (100 = average)")
ax.set_title("How Negative, Neutral and Positive posts differ", pad=30)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0))
plt.show()

**What to look for:** Neutral posts are much **shorter**, use fewer `!` and negations, and have a **higher CAPS ratio**
("BREAKING:", "US", "FM" …): they are mostly plain news headlines.
Negative and Positive posts look alike in style, so style helps to separate *Neutral vs. opinion*, while the words
(TF-IDF) are needed to tell *Negative vs. Positive*.

## 4.6 Sentiment model: how Logistic Regression uses the features
> 🧠 **The logic.** Logistic Regression learns a **weight** for every feature and every class.
> For a post it multiplies each feature value by its weight and adds them up, giving **one score per class**.
> The scores become probabilities (softmax), and the class with the highest probability wins.
> Because it is just "value × weight", we can see exactly *why* it chose a label (4.6.2).

- **Target:** the VADER label. We have no human labels, so VADER acts as the "teacher" (*weak supervision*).
- **Train / test split:** learn from 80% of posts, check on the 20% it has never seen, to test understanding, not memory.
- `class_weight="balanced"`: Neutral posts are fewer, so mistakes on them count more. Otherwise the model would ignore them.
- **Two versions:** (A) TF-IDF only vs. (B) TF-IDF + hand-crafted features, to see what feature engineering adds.
- The scaler is fitted on the training posts only, so nothing from the test posts leaks into training.

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

y = df["vader_label"]
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# Scale hand-crafted features (fit on training posts only), then glue them next to the TF-IDF columns
scaler = StandardScaler().fit(features.iloc[train_idx])
X_combined = hstack([X, csr_matrix(scaler.transform(features))]).tocsr()

results, models = [], {}
for name, matrix in [("A. TF-IDF only", X), ("B. TF-IDF + hand-crafted features", X_combined)]:
    model = LogisticRegression(max_iter=2000, class_weight="balanced")
    model.fit(matrix[train_idx], y.iloc[train_idx])
    predicted = model.predict(matrix[test_idx])
    models[name] = model
    results.append({"model": name,
                    "accuracy": f"{accuracy_score(y.iloc[test_idx], predicted):.1%}",
                    "macro F1": round(f1_score(y.iloc[test_idx], predicted, average="macro"), 3),
                    "Neutral F1": round(f1_score(y.iloc[test_idx], predicted, labels=["Neutral"], average="macro"), 3)})
pd.DataFrame(results)

The point here is the *comparison* (same model, with vs. without engineered features), not the exact score.
*Accuracy* = % of test posts labelled correctly. *F1* balances precision (when the model says
"Neutral", is it right?) and recall (does it find all Neutral posts?). *Macro F1* averages the three classes equally.

The hand-crafted features mainly help with **Neutral** posts, as the chart in 4.5 suggested. The gain is modest because
both models learn from the same teacher (VADER) and can only ever be as good as it is. Part C measures how good that teacher is.

In [ ]:
# Which hand-crafted features matter? (weights of model B for the engineered columns)
model_b = models["B. TF-IDF + hand-crafted features"]
feature_weights = pd.DataFrame(model_b.coef_[:, -features.shape[1]:].T,
                               index=features.columns, columns=model_b.classes_)[SENTIMENTS]
feature_weights.round(2).style.background_gradient(cmap=BLUES, axis=None)

A positive weight pushes a post *towards* that class. For example, a large negative weight for `length_chars` under
Neutral means "the longer the post, the less likely it is Neutral".

### 4.6.2 Explaining one prediction
Let's open the "black box" for a single real post: which words and features pushed it to its label?

In [ ]:
model_b = models["B. TF-IDF + hand-crafted features"]
feature_names = np.array(list(terms) + list(features.columns))

i = test_idx[np.argmax(df["score"].to_numpy()[test_idx])]   # the most-liked post the model never saw in training
row = X_combined[i]
probabilities = model_b.predict_proba(row)[0]
predicted = model_b.classes_[probabilities.argmax()]

# contribution of each feature = feature value × its weight for the predicted class
weights = model_b.coef_[list(model_b.classes_).index(predicted)]
contributions = pd.Series(row.multiply(weights).toarray().ravel(), index=feature_names)
top = contributions[contributions != 0].sort_values(key=abs, ascending=False).head(8)

print("Post:", df.loc[i, "text"][:300])
print("\nProbabilities:", {c: f"{p:.0%}" for c, p in zip(model_b.classes_, probabilities)})
print(f"Prediction: {predicted}   (VADER label: {df.loc[i, 'vader_label']})\n")
print(f"Top reasons for '{predicted}' (value × weight; positive = pushes towards {predicted}):")
print(top.round(3).to_string())

To label **every** post for the comparison in Part C, we use model A with **5-fold cross-validation**: each post is
predicted by a model that did not see it during training.

In [ ]:
sentiment_model = models["A. TF-IDF only"]
df["tfidf_label"] = cross_val_predict(
    LogisticRegression(max_iter=1000, class_weight="balanced"), X, y, cv=5)
df["tfidf_label"].value_counts().reindex(SENTIMENTS)

In [ ]:
# Which words push a post towards Negative or Positive? (the model's learned weights)
classes = list(sentiment_model.classes_)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, label in zip(axes, ["Negative", "Positive"]):
    weights = pd.Series(sentiment_model.coef_[classes.index(label)], index=terms).nlargest(12)
    ax.barh(weights.index, weights.values, color=SENT_COLORS[label], height=0.65)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_title(f"Words that signal {label}")
    ax.set_xlabel("model weight")
plt.tight_layout()
plt.show()

**Look closely at the Positive list.** If phrases about talks *"reaching a plateau"* show up there, that is not because a
stalled deal is good news. VADER scored many of those posts as positive (because of words like "giving" or
"relief"), and the model copied its teacher. This is the main weakness of weak supervision, and Part C measures it.

## 4.7 Topics with NMF (what are people talking about?)
**NMF (Non-negative Matrix Factorization)** finds groups of words that tend to appear together in the same posts.
Each group is a **topic**. We pick 8 topics, and each post is assigned to its strongest topic.
Topics are named after their top 3 keywords.

> 🧠 **The logic.** Our TF-IDF table is *posts × words*. NMF splits it into two smaller tables:
> *posts × topics* (how much each post is about each topic) and *topics × words* (which words make up each topic),
> chosen so that multiplying them back gives roughly the original table. "Non-negative" means no negative numbers,
> so a topic can only *add* words, which keeps topics readable. Nobody tells NMF what the topics are: it discovers
> them from which words keep appearing together.

In [ ]:
N_TOPICS = 8
nmf = NMF(n_components=N_TOPICS, init="nndsvda", max_iter=800, random_state=RANDOM_STATE)
post_topic_weights = nmf.fit_transform(X_topics)   # rows = posts, columns = topics

topic_names = [f"T{i + 1}: " + " / ".join(topic_terms[w.argsort()[::-1][:3]]) for i, w in enumerate(nmf.components_)]
topic_table = pd.DataFrame({
    "topic": topic_names,
    "top keywords": [", ".join(topic_terms[w.argsort()[::-1][:8]]) for w in nmf.components_],
})

# Posts that contain none of the TF-IDF vocabulary cannot be assigned to a topic
has_keywords = post_topic_weights.sum(axis=1) > 0
df["nmf_topic"] = np.where(has_keywords, np.array(topic_names)[post_topic_weights.argmax(axis=1)],
                           "No clear keywords")
topic_table["posts"] = topic_table["topic"].map(df["nmf_topic"].value_counts())
topic_table

In [ ]:
plot_barh(df["nmf_topic"].value_counts(), "How many posts fall into each NMF topic", "posts")
plot_sentiment_shares(share_table(df, "nmf_topic", "vader_label"), "Sentiment (VADER) within each topic")
plt.show()

In [ ]:
# Read two real posts per topic: this is how we check that a topic means something
for topic in topic_names:
    print(f"■ {topic}")
    for text in df[df["nmf_topic"] == topic].nlargest(2, "score")["text"]:
        print("   –", text[:200].replace("\n", " "))
    print()

## 4.8 Sentiment over time
**Net sentiment = % Positive − % Negative** for each day (only days with at least 20 posts).
Above 0 means more positive than negative posts that day.

In [ ]:
def daily_net_sentiment(data, label_col, min_posts=20):
    day = data["created_utc"].dt.floor("D")
    shares = pd.crosstab(day, data[label_col], normalize="index").reindex(columns=SENTIMENTS, fill_value=0)
    net = (shares["Positive"] - shares["Negative"]) * 100
    return net[day.value_counts().reindex(net.index) >= min_posts]

net_vader = daily_net_sentiment(df, "vader_label")
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(net_vader.index, net_vader.values, color=METHOD_COLORS["VADER"], linewidth=2, marker="o", markersize=5)
ax.axhline(0, color=MUTED, linewidth=1)
ax.set_title("Daily net sentiment (VADER)")
ax.set_ylabel("% positive − % negative")
fig.autofmt_xdate()
plt.show()

### ✅ Part A takeaways
- **Keywords.** TF-IDF and NMF surface clear storylines without anyone reading the posts: talks hitting a
  *"plateau"*, *Russian oil* and sanctions, USTR *Greer* / minister *Goyal* negotiations, *pharma* and *steel*.
- **Feature engineering.** TF-IDF features say *what* a post is about; hand-crafted features (length, `!`, negations…)
  say *how* it is written. Adding them mainly helps to recognise Neutral, headline-style posts.
- **Sentiment.** VADER splits the conversation between Positive and Negative, with few Neutral posts.
  Keep this in mind: VADER scores *words*, not *meaning*. A headline such as "BIG RELIEF FOR INDIA?" looks positive to VADER even when it is a question.
- **Limits.** One NMF topic is a large catch-all. Posts with no distinctive keyword land there, and NMF cannot tell
  a sarcastic post or an off-topic post (for example, about Canadian steel) from a genuine one. This is where an LLM should help.

---
# Part B — The same analysis with an LLM (Groq API)
Instead of counting words, we **ask a Large Language Model to read every post** and return, in a fixed JSON format:

| field | values |
|---|---|
| `sentiment` | Positive / Neutral / Negative (the author's *real* tone, so sarcasm is read correctly) |
| `topic` | one of 6 topics defined below |
| `sarcastic` | true / false |
| `reason` | ≤ 12 words explaining the label |

**Why Groq?** Groq runs open-source LLMs (Llama, GPT-OSS, …) on special chips, so answers come back very fast, and it
has a **free tier**.

> 🧠 **The logic.** An LLM has learned language from a huge amount of text, so it reads *meaning and context*, not
> single words. We turn it into a reliable classifier with four design choices:
> 1. **A fixed label set** (3 sentiments, 6 topics) → answers we can count and compare.
> 2. **Clear rules for hard cases** (sarcasm, plain news) → consistent decisions.
> 3. **JSON output** → Python can read the answers directly.
> 4. **Batches + pacing + saving** → stays inside the free limits and never loses progress.

## 5.1 Free Groq API key
1. Go to **console.groq.com** → sign in → **API Keys** → **Create API Key** and copy it.
2. In Colab, open **🔑 Secrets** (left sidebar), add a secret named `GROQ_API_KEY` with the key as its value, and switch on
   **Notebook access**. Otherwise the cell below asks you to paste the key (it stays hidden).

**Free, with limits.** The free tier limits how many **tokens per minute** you can use (a token ≈ ¾ of a word).
Labelling all ~1,700 posts uses roughly 200,000 tokens, so the notebook paces itself and takes about **35–40 minutes**.
Progress is saved after every request. Set `LLM_LIMIT = 300` for a quick ~7-minute test run.

In [ ]:
from groq import Groq, RateLimitError, APIStatusError, AuthenticationError, NotFoundError

if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"], max_retries=0)   # we handle waiting ourselves

available = sorted(model.id for model in client.models.list().data)
print("Models your key can use:", ", ".join(available))

### Choose the model
| Model | Size | Free daily allowance | Use it when |
|---|---|---|---|
| `llama-3.1-8b-instant` | 8B (small, fast) | largest (whole dataset fits in one day) | **default** |
| `openai/gpt-oss-20b` | 20B | smaller | you want better reading of sarcasm and can wait / split over days |
| `llama-3.3-70b-versatile` | 70B (large) | smallest | a small, high-quality sample (`LLM_LIMIT = 300`) |

Limits change over time; your exact limits are shown at **console.groq.com → Settings → Limits**.

In [ ]:
MODEL = "llama-3.1-8b-instant"
TOKENS_PER_MINUTE = 6000   # free-tier limit for this model (check console.groq.com → Settings → Limits)

if MODEL not in available:
    fallback = next((m for m in ["llama-3.1-8b-instant", "openai/gpt-oss-20b", "llama-3.3-70b-versatile"] if m in available),
                    available[0])
    print(f"{MODEL} is not available for this key, using {fallback} instead.")
    MODEL = fallback
print("Using:", MODEL)

## 5.2 The prompt
A good classification prompt is **specific**. It defines every label, says how to treat tricky cases (sarcasm,
plain news) and fixes the output format. The topics below come from reading the posts and from the NMF results in Part A.

In [ ]:
BATCH_SIZE = 30               # posts per request
LLM_LIMIT = None              # e.g. 300 for a quick test run; None = all posts
LLM_CACHE = "llm_labels.csv"  # results are saved here, so a re-run continues instead of starting over

TOPICS = {
    "Deal negotiations": "progress or stalling of the trade talks, interim deal, statements by negotiators",
    "Russian oil & sanctions": "India buying Russian oil, the US sanctions law, 100% tariff threats on oil buyers",
    "Tariffs & economic impact": "tariff rates and their effect on exports, sectors (pharma, steel, farming, IT), markets, jobs",
    "Geopolitics & diplomacy": "strategic ties, China, Modi–Trump relationship, diplomatic meetings",
    "Domestic politics": "praise or criticism of either government, opposition attacks, nationalism",
    "Not about India–US trade": "off-topic, or India–US trade is only mentioned in passing",
}

topic_lines = "\n".join(f"  - {name}: {description}" for name, description in TOPICS.items())
SYSTEM_PROMPT = (
    "You label social-media posts about the India–US trade deal for a research project.\n\n"
    "For every post return:\n"
    "- sentiment: the author's real overall tone — Positive, Negative or Neutral.\n"
    "  Read sarcasm and mockery as what the author actually means.\n"
    "  Plain news that only reports facts, without the author's opinion, is Neutral.\n"
    "- topic: the single best-fitting topic, using the exact name from this list:\n"
    f"{topic_lines}\n"
    "- sarcastic: true if the post is sarcastic or mocking, else false.\n"
    "- reason: at most 12 words explaining the sentiment.\n\n"
    "Answer with JSON only, in exactly this format, with one result per post:\n"
    '{"results": [{"id": "<post id>", "sentiment": "Positive", "topic": "<topic name>", '
    '"sarcastic": false, "reason": "<short reason>"}]}'
)
print(SYSTEM_PROMPT)

## 5.3 Label all posts
Each request sends a batch of 30 posts and gets back 30 labels.

- `response_format={"type": "json_object"}` makes the model answer with **valid JSON**. We still check every label
  in Python and drop anything that is not one of our allowed sentiments or topics.
- `temperature=0` makes the answers as consistent as possible between runs.
- **Pacing:** after each request we wait just long enough to stay under the tokens-per-minute limit
  (the API tells us how many tokens each request used).
- If Groq says "too many requests" (error 429), we wait as long as it asks and retry. If a **daily** limit is used up,
  we stop and keep everything labelled so far: run the cell again later and it continues where it stopped.

In [ ]:
import time

class DailyLimitReached(Exception):
    pass

def label_batch(batch, retries=4):
    # Send one batch of posts to the LLM. Returns (list of valid labels, tokens used).
    posts = "\n".join(json.dumps({"id": r.id, "text": r.text}, ensure_ascii=False) for r in batch.itertuples())
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Label these {len(batch)} posts (one JSON object per line):\n{posts}"},
                ],
                response_format={"type": "json_object"},
                temperature=0,
                max_completion_tokens=3000,
            )
            break
        except RateLimitError as error:
            if "per day" in str(error).lower():
                raise DailyLimitReached(str(error)[:200])
            wait = float(error.response.headers.get("retry-after", 30)) + 1
            print(f"\n  rate limit hit, waiting {wait:.0f} s ...")
            time.sleep(wait)
    else:
        raise RuntimeError("still rate-limited after retries")

    results = json.loads(response.choices[0].message.content).get("results", [])
    batch_ids = set(batch["id"])
    valid = [{**r, "id": str(r.get("id"))} for r in results
             if str(r.get("id")) in batch_ids and r.get("sentiment") in SENTIMENTS and r.get("topic") in TOPICS]
    return valid, response.usage.total_tokens


def save(labels):
    labels.drop_duplicates("id", keep="last").to_csv(LLM_CACHE, index=False)


# Which posts still need labels? (skip the ones already saved)
to_label = df if LLM_LIMIT is None else df.sample(LLM_LIMIT, random_state=RANDOM_STATE)
llm_labels = pd.read_csv(LLM_CACHE, dtype={"id": str}) if os.path.exists(LLM_CACHE) else pd.DataFrame()
if len(llm_labels):
    to_label = to_label[~to_label["id"].isin(llm_labels["id"])]

batches = [to_label.iloc[i:i + BATCH_SIZE] for i in range(0, len(to_label), BATCH_SIZE)]
print(f"{len(to_label):,} posts to label in {len(batches)} requests")

failures_in_a_row = 0
for number, batch in enumerate(batches, start=1):
    try:
        labels, tokens_used = label_batch(batch)
        llm_labels = pd.concat([llm_labels, pd.DataFrame(labels)], ignore_index=True)
        save(llm_labels)   # save after every request, so nothing is lost
        failures_in_a_row = 0
        print(f"\r  {number}/{len(batches)} requests done ({len(labels)}/{len(batch)} labels kept)", end="")
        time.sleep(tokens_used / TOKENS_PER_MINUTE * 60)   # pacing: stay under the tokens-per-minute limit
    except DailyLimitReached:
        print("\n⚠️ Daily free limit reached. Progress is saved: run this cell again later to continue,")
        print("   or set MODEL to another model from the list in 5.1 and run it now.")
        break
    except (AuthenticationError, NotFoundError) as error:
        print(f"\n❌ Setup problem: {error}. Check your API key and MODEL name.")
        break
    except Exception as error:   # one bad batch (e.g. broken JSON) should not stop the whole run ...
        print(f"\n  batch {number} failed: {str(error)[:150]}")
        failures_in_a_row += 1
        if failures_in_a_row == 3:   # ... but three in a row means something is wrong
            print("❌ 3 batches failed in a row, stopping. Check the error above, or try another MODEL.")
            break

print(f"\nLabels saved for {llm_labels['id'].nunique() if len(llm_labels) else 0:,} posts → {LLM_CACHE}")

In [ ]:
# Join the LLM labels onto our posts (keep only ids that really exist in our data)
if "sentiment" not in llm_labels.columns:
    raise RuntimeError("No LLM labels yet: fix the problem shown in the cell above, then run it again.")
llm_labels = llm_labels.rename(columns={"sentiment": "llm_label", "topic": "llm_topic", "reason": "llm_reason"})
labeled = df.merge(llm_labels[["id", "llm_label", "llm_topic", "sarcastic", "llm_reason"]], on="id", how="inner")
labeled["sarcastic"] = labeled["sarcastic"].astype(str).str.lower() == "true"

print(f"{len(labeled):,} of {len(df):,} posts have LLM labels")
labeled[["text", "llm_label", "llm_topic", "sarcastic", "llm_reason"]].sample(5, random_state=RANDOM_STATE)

## 5.4 What the LLM found

In [ ]:
counts = labeled["llm_label"].value_counts().reindex(SENTIMENTS, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(counts.index, counts.values, color=[SENT_COLORS[s] for s in counts.index], width=0.6)
for x, v in enumerate(counts.values):
    ax.text(x, v, f"{v:,}\n({v / len(labeled):.0%})", ha="center", va="bottom", fontsize=10, color=INK2)
ax.set_title("LLM sentiment of all posts")
ax.set_ylabel("posts")
ax.grid(axis="x", visible=False)
ax.margins(y=0.2)
plt.show()

print(f"Sarcastic or mocking posts : {labeled['sarcastic'].mean():.0%}")
print(f"Not really about the deal  : {(labeled['llm_topic'] == 'Not about India–US trade').mean():.0%}")

In [ ]:
plot_barh(labeled["llm_topic"].value_counts(), "LLM topics: what are people talking about?", "posts")
plot_sentiment_shares(share_table(labeled, "llm_topic", "llm_label"), "Sentiment (LLM) within each topic")
plt.show()

## 5.5 LLM-written summary of the community conversation
Finally, we give the LLM the numbers from 5.4 plus the most-liked posts in each topic, and ask for a short
report. The model only sees real data from this notebook, and it is told not to invent anything.

In [ ]:
def topic_digest(data, per_topic=8):
    # Build a compact text summary: stats + top posts per topic.
    parts = []
    on_topic = data[data["llm_topic"] != "Not about India–US trade"]
    for topic, group in on_topic.groupby("llm_topic"):
        shares = group["llm_label"].value_counts(normalize=True).reindex(SENTIMENTS, fill_value=0)
        stats = ", ".join(f"{s} {shares[s]:.0%}" for s in SENTIMENTS)
        posts = "\n".join(f"- ({r.score} likes) {r.text[:220]}" for r in group.nlargest(per_topic, "score").itertuples())
        parts.append(f"## {topic} — {len(group)} posts ({stats})\n{posts}")
    return "\n\n".join(parts)

digest = topic_digest(labeled)
period = f"{df['created_utc'].min():%d %b} – {df['created_utc'].max():%d %b %Y}"

report_prompt = f'''Below are posts from X about the India–US trade deal ({period}),
grouped by topic with sentiment shares and the most-liked posts.

Write a short report in Markdown for a class presentation:
1. One-paragraph overview of the mood.
2. "What people are talking about": the 5 most important themes, each with 2–3 sentences,
   the sentiment, and one short quote (under 20 words).
3. "What drives negative sentiment" and "What drives positive sentiment" (3 bullets each).
4. "Surprises": 2 bullets.
Keep it under 450 words.

{digest}'''

for attempt in range(3):
    try:
        report = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a social-media analyst. Use only the data you are given. "
                                              "Do not invent facts, numbers or quotes."},
                {"role": "user", "content": report_prompt},
            ],
            temperature=0.3,
            max_completion_tokens=1200,
        )
        llm_report = report.choices[0].message.content
        break
    except RateLimitError as error:   # wait for the per-minute token budget to refill
        wait = float(error.response.headers.get("retry-after", 60)) + 1
        print(f"rate limit hit, waiting {wait:.0f} s ...")
        time.sleep(wait)
else:
    llm_report = "_Report not generated (rate limit). Run this cell again in a minute._"
display(Markdown(llm_report))

---
# Part C — Comparison: TF-IDF vs. LLM
> 🧠 **The logic.** There is no "answer key" for these posts. So we compare: where the methods agree we can be more
> confident, and where they disagree, reading the posts shows which logic (counting words vs. reading meaning) works better.

We compare on the posts that have labels from all methods:

- **VADER**: word dictionary
- **TF-IDF model**: Logistic Regression trained on VADER labels
- **LLM**: a Llama model (via Groq) reading each post

## 6.1 Do they see the same overall mood?

In [ ]:
METHODS = {"VADER": "vader_label", "TF-IDF model": "tfidf_label", "LLM": "llm_label"}
overall = pd.DataFrame({name: labeled[col].value_counts(normalize=True) * 100 for name, col in METHODS.items()}).T
overall = overall.reindex(columns=SENTIMENTS, fill_value=0)
plot_sentiment_shares(overall, "Overall sentiment by method")
plt.show()
overall.round(1)

## 6.2 How often do they agree?
- **Agreement** = % of posts where both methods give the same label.
- **Cohen's kappa** = agreement after removing the agreement that would happen by pure chance:
  `kappa = (observed agreement − chance agreement) ÷ (1 − chance agreement)`.
  If two methods both say "Positive" to most posts, they agree a lot just by luck; kappa corrects for that.
  Roughly: < 0.2 slight, 0.2–0.4 fair, 0.4–0.6 moderate, > 0.6 substantial.

We treat the LLM as the closest thing to a human reader, so its labels are the reference.

In [ ]:
pairs = [("VADER", "LLM"), ("TF-IDF model", "LLM"), ("VADER", "TF-IDF model")]
agreement = pd.DataFrame([{
    "comparison": f"{a} vs {b}",
    "agreement": f"{(labeled[METHODS[a]] == labeled[METHODS[b]]).mean():.0%}",
    "Cohen's kappa": round(cohen_kappa_score(labeled[METHODS[a]], labeled[METHODS[b]]), 2),
} for a, b in pairs])
agreement

In [ ]:
# Confusion matrices: rows = classic method, columns = LLM. The diagonal = agreement.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, method in zip(axes, ["VADER", "TF-IDF model"]):
    matrix = confusion_matrix(labeled[METHODS[method]], labeled["llm_label"], labels=SENTIMENTS)
    ax.imshow(matrix, cmap=BLUES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", fontsize=12,
                    color="white" if matrix[i, j] > matrix.max() * 0.55 else INK)
    ax.set_xticks(range(3), SENTIMENTS)
    ax.set_yticks(range(3), SENTIMENTS)
    ax.set_xlabel("LLM label")
    ax.set_ylabel(f"{method} label")
    ax.set_title(f"{method} vs LLM")
    ax.grid(False)
plt.tight_layout()
plt.show()

## 6.3 Where do they disagree, and who is right?
Reading the disagreements is the most useful part of a comparison. Below: posts that **VADER calls Positive but
the LLM calls Negative**, and the reverse. Read the LLM's `reason` column and decide for yourself.

In [ ]:
cols = ["text", "vader_score", "llm_label", "sarcastic", "llm_reason"]
print("VADER = Positive, LLM = Negative")
display(labeled[(labeled["vader_label"] == "Positive") & (labeled["llm_label"] == "Negative")].nlargest(6, "score")[cols])
print("VADER = Negative, LLM = Positive")
display(labeled[(labeled["vader_label"] == "Negative") & (labeled["llm_label"] == "Positive")].nlargest(4, "score")[cols])

In [ ]:
# How well does VADER do on sarcastic posts compared with normal ones?
sarcasm_check = labeled.groupby("sarcastic").apply(
    lambda g: pd.Series({"posts": len(g), "VADER agrees with LLM": f"{(g['vader_label'] == g['llm_label']).mean():.0%}"}),
    include_groups=False)
sarcasm_check.index = sarcasm_check.index.map({False: "Not sarcastic", True: "Sarcastic"})
sarcasm_check

## 6.4 Do the topics match?
Each row is an NMF topic (Part A). The colours show which LLM topic its posts were given (row %). A row that is dark in
one column means both methods found the same theme. A row spread across columns means the NMF topic is mixed.

In [ ]:
topic_match = pd.crosstab(labeled["nmf_topic"], labeled["llm_topic"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(12, 0.55 * len(topic_match) + 2))
ax.imshow(topic_match.values, cmap=BLUES, aspect="auto", vmin=0, vmax=100)
for i in range(topic_match.shape[0]):
    for j in range(topic_match.shape[1]):
        v = topic_match.values[i, j]
        if v >= 5:
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center", fontsize=9, color="white" if v > 55 else INK)
ax.set_xticks(range(topic_match.shape[1]), topic_match.columns, rotation=25, ha="right")
ax.set_yticks(range(topic_match.shape[0]), topic_match.index)
ax.set_title("NMF topic (rows) → LLM topic (columns), % of each NMF topic")
ax.grid(False)
plt.show()

## 6.5 Sentiment over time, both methods

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for method in ["VADER", "LLM"]:
    net = daily_net_sentiment(labeled, METHODS[method])
    ax.plot(net.index, net.values, color=METHOD_COLORS[method], linewidth=2, marker="o", markersize=5, label=method)
ax.axhline(0, color=MUTED, linewidth=1)
ax.set_title("Daily net sentiment: VADER vs LLM")
ax.set_ylabel("% positive − % negative")
ax.legend(loc="upper left")
fig.autofmt_xdate()
plt.show()

## 6.6 Side-by-side summary

| | **TF-IDF + VADER + Logistic Regression** | **LLM (Groq)** |
|---|---|---|
| How it works | Counts words; a dictionary scores them | Reads and interprets each post |
| Sarcasm, irony | ❌ Takes words literally | ✅ Reads the real meaning |
| News vs. opinion | ❌ "BIG RELIEF?" scores as positive | ✅ Plain news is labelled Neutral |
| Off-topic posts | ❌ Cannot tell | ✅ Flags them |
| Hinglish / slang | ⚠️ Unknown words are ignored | ✅ Mostly understood |
| Topics | Keyword groups, one big catch-all | Named, meaningful categories |
| Explainability | ✅ Exact word weights | ⚠️ A short reason per post |
| Speed & cost | ✅ Seconds, free, no limits | ⚠️ Minutes; free tier has daily request limits |
| Reproducibility | ✅ Same result every run | ⚠️ Can vary slightly between runs |

**Bottom line:** TF-IDF is fast, free and transparent, which makes it great for *finding keywords and storylines*.
The LLM is better at *understanding what people mean*, which matters for sentiment on political social media,
where sarcasm and news headlines are common.

---
## 7. Insights: what is the community talking about?
The findings below come from reading the TF-IDF topics, the most-liked posts and the LLM's labels. The cell under them counts the evidence.

**1. The talks have stalled, and that is the headline.**
The most-shared news post is Finance Minister Nirmala Sitharaman saying the talks have reached a *"plateau"*,
where further give-and-take "might be very very difficult". USTR Jamieson Greer added that a deal is *"not imminent"*,
pointing to unresolved "sticking points". Minister Piyush Goyal and Greer met at the G20 trade meeting in Milwaukee to push for
an **interim agreement**. Most of these posts are factual news, and the opinions around them are pessimistic.

**2. Russian oil is the biggest pressure point.**
A new US Russia-sanctions law lets the President put **tariffs of up to 100%** on countries that buy Russian energy.
People discuss the Modi–Trump phone call, Secretary of State Marco Rubio pointing to *national-security waivers*
(framed as "relief for India"), shipping data showing Russia stayed India's top crude supplier, and JPMorgan CEO
Jamie Dimon saying the US "should not punish India". The tone is tense and often defiant.

**3. Sovereignty and nationalism split the conversation.**
One side reads the stalemate as India standing firm ("US pressure fails"), and some posts are openly anti-American.
The other side accuses the government of weakness ("vassal state", "surrender"). Posts praising Indian negotiators
sit next to sarcastic attacks on them. This is where sarcasm is most common, and where VADER struggles most.

**4. Sector stories bring the few clear positives.**
The **exemption of some Indian speciality drugs from 100% pharma tariffs** (with Sun Pharma and Aurobindo stocks in focus)
is one of the few clearly positive storylines. Steel duties and market reactions also come up.

**5. Side stories get mixed into the trade debate.**
Elon Musk's criticism of India and the **Starlink** security debate, comparisons with **China**, and External Affairs Minister
Jaishankar's remark that the US has become "a lonelier power" all come up in the same conversation.

**6. Not every post that matches the keywords is really about the deal.**
Canadian steel politics and US domestic debates mention India only in passing. Keyword methods count them, while the LLM
flags them as off-topic. This is one practical reason to use an LLM for this kind of analysis.

In [ ]:
# Evidence: how many cleaned posts mention each storyline?
storylines = {
    "Talks stalled (plateau, stuck, not imminent)": r"plateau|stall|stuck|not imminent|sticking point|hit a wall",
    "Russian oil / sanctions": r"russian oil|russia\W.{0,20}oil|crude|sanction",
    "100% tariff threat": r"100 ?%",
    "Interim deal": r"interim",
    "Pharma / drugs": r"pharma|drug|medicine|generic",
    "Starlink / Musk": r"starlink|musk|elon",
    "China": r"china|chinese",
    "Farm / dairy": r"agri|dairy|farmer",
}
text_lower = df["text"].str.lower()
evidence = pd.Series({name: text_lower.str.contains(pattern).sum() for name, pattern in storylines.items()})
plot_barh(evidence.sort_values(ascending=False), "Posts mentioning each storyline", "posts")

---
## 8. Conclusion

**What we did.** We collected real X posts on the US–India trade deal, cleaned them down to distinct voices, and
analysed them twice: once with classic NLP (TF-IDF, VADER, Logistic Regression, NMF) and once with an LLM.

**What we found.**
- The conversation is dominated by **stalled talks** and the **Russian-oil tariff threat**. Mood is cautious-to-negative,
  with positive spikes around sector exemptions and diplomatic "relief" headlines.
- Classic NLP is **excellent for discovering keywords and storylines** quickly and for free.
- For **sentiment**, the LLM reads meaning (sarcasm, news vs. opinion, off-topic posts), so it disagrees with the
  word-counting approach on a large share of posts. The disagreement examples in 6.3 show why.

**Limitations.**
- Only X data, over about 3 weeks, from English posts found by keyword search. This is not a representative sample of all opinion.
- The TF-IDF model learns from VADER labels, so it inherits VADER's mistakes.
- We have no human-labelled "ground truth". The LLM is a strong reader, but it is not perfect.

**Next steps.** Hand-label ~200 posts to measure true accuracy of both methods. Add Reddit for a second community.
Train the TF-IDF model on LLM labels instead of VADER labels (a cheap model learning from a smart teacher).

In [ ]:
# Save everything for the report / slides
out_cols = ["id", "created_utc", "text", "score", "vader_score", "vader_label", "tfidf_label", "nmf_topic"]
df[out_cols].to_csv("posts_classic_nlp.csv", index=False)
if "labeled" in globals():
    labeled.to_csv("posts_with_all_labels.csv", index=False)
    with open("llm_community_report.md", "w") as f:
        f.write(llm_report)
print("Saved: posts_classic_nlp.csv, posts_with_all_labels.csv, llm_community_report.md")

try:   # download in Colab
    from google.colab import files
    for name in ["posts_with_all_labels.csv", "llm_community_report.md"]:
        files.download(name)
except ImportError:
    pass